# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'external brand site',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 14 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'offerings page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'offerings page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type'

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community page', 'url': 'https://discuss.huggingface.co'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discord community', 'url': 'https://huggingface.co/join/discord'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 26 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-35B-A3B
Updated
1 day ago
•
378k
•
676
Qwen/Qwen3.5-27B
Updated
4 days ago
•
172k
•
434
unsloth/Qwen3.5-35B-A3B-GGUF
Updated
about 24 hours ago
•
350k
•
358
Qwen/Qwen3.5-122B-A10B
Updated
4 days ago
•
120k
•
344
Qwen/Qwen3.5-397B-A17B
Updated
5 days ago
•
889k
•
1.13k
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.74k
Qwen Image Multiple Angles 3D Camera
🎥
1.74k
Change the camera angle of a photo with AI
Running
on
Zero
Featured
231
Omni Video Factory
🏆
231
text to video, image to video, video extend
Running
on
Zero

In [21]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\n1 day ago\n•\n378k\n•\n676\nQwen/Qwen3.5-27B\nUpdated\n4 days ago\n•\n172k\n•\n434\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\n1 day ago\n•\n350k\n•\n358\nQwen/Qwen3.5-122B-A10B\nUpdated\n4 days ago\n•\n120k\n•\n344\nQwen/Qwen3.5-397B-A17B\nUpdated\n5 days ago\n•\n889k\n•\n1.13k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.74k\nQwen Image Multiple An

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Company Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. It is a collaborative platform where data scientists, developers, and researchers worldwide come together to create, share, and improve machine learning models, datasets, and applications. Known as "The Home of Machine Learning," Hugging Face empowers users to accelerate AI innovation through open-source technologies and community-driven efforts.

---

## Our Platform

- **Models**: Access and contribute to over **2 million machine learning models**, including trending and state-of-the-art models like Qwen3.5 series.
- **Datasets**: Discover and use from a growing library of **500,000+ datasets**, supporting a variety of ML tasks.
- **Spaces**: Experiment and showcase custom ML applications running on Hugging Face servers.
- **Applications**: Explore over **1 million AI applications** built by the community.

The platform supports all AI modalities, including **text, image, video, audio, and even 3D**, enabling diverse and innovative AI solutions.

---

## For Developers & ML Practitioners

- **Collaboration**: Host unlimited public models, datasets, and applications for free.
- **Build your portfolio**: Share your work and enhance your machine learning profile visible to the global AI community.
- **Open Source**: Leverage the Hugging Face open-source stack to move faster in ML development.
- **Compute & Enterprise Solutions**: Access scalable paid compute resources and enterprise tools to accelerate AI projects efficiently.

---

## Enterprise Solutions

Hugging Face offers tailored Enterprise and Team solutions designed to scale AI development securely within organizations:

- **Enterprise-grade security** with Single Sign-On (SSO) integration.
- **Access controls** and **dedicated support**.
- Regulatory compliance with **region management** and audit logs.
- Flexible contracts starting at $20/user/month for Teams, with customizable Enterprise plans.

These offerings help companies securely manage their AI initiatives with robust governance and performance needs.

---

## Company Culture

Hugging Face thrives on **community, openness, and collaboration**. It is a vibrant ecosystem where innovation is driven by the collective intelligence of thousands of AI enthusiasts and experts worldwide.

- **Community Focused**: Encourages knowledge sharing, mutual help, and co-creation.
- **Inclusive and Diverse**: Welcomes contributors from all backgrounds and skill levels.
- **Transparent and Open Source**: Committed to democratizing AI and fostering open access to resources.

---

## Careers and Opportunities

Join Hugging Face to be part of an **industry-leading AI company** shaping the future of technology:

- Work with cutting-edge machine learning tools and models.
- Collaborate with top-tier AI researchers and engineers globally.
- Engage in meaningful projects that impact diverse sectors.
- Competitive salaries, inclusive culture, and opportunities for growth.

To explore current openings and join the team, visit their careers page on the Hugging Face website.

---

## Why Choose Hugging Face?

- **Largest Open ML Community**: Connect with millions of AI practitioners and developers.
- **Comprehensive AI Platform**: From models and datasets to compute and enterprise solutions.
- **Robust Security and Scalability**: Enterprise offerings built for modern business needs.
- **Cross-Modality AI Support**: Text, images, video, audio, and 3D all supported.
- **Accelerate AI Innovation**: Open-source tools and community collaboration fuel faster development.

---

## Get Started Today!

Sign up for free or explore enterprise offerings at [huggingface.co](https://huggingface.co) and become part of the AI community building the future.

---

*Hugging Face – Where innovation meets collaboration in AI.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. It serves as a vibrant collaboration platform where engineers, scientists, and AI enthusiasts from around the world create, explore, and share machine learning models, datasets, and applications. With over 2 million models and more than 500,000 datasets available, Hugging Face empowers the next generation of AI developers and users to build an open and ethical AI future together.

At its core, Hugging Face is about **community, collaboration, and innovation**—making ML accessible and accelerating advancements across all AI modalities including text, image, video, audio, and even 3D.

---

## What We Offer

### Open Source Collaboration Hub

- **Models:** Browse and contribute to a vast repository of pre-trained machine learning models, from cutting-edge large language models to domain-specific applications.
- **Datasets:** Access and share hundreds of thousands of datasets to fuel your AI projects, including specialized collections for reasoning, coding, and visual understanding.
- **Spaces:** Host machine learning apps and demos easily on Hugging Face Spaces, fostering rapid experimentation and sharing with interactive, live applications.
- **Community:** Join a fast-growing, diverse AI community collaborating openly on research, engineering, and product innovation.

### Enterprise Solutions

Hugging Face offers an advanced platform tailored to organizations, enabling teams to securely build and scale AI workflows with enterprise-grade features:

- **Security & Compliance:** Single Sign-On (SSO), audit logs, granular access controls, and private storage ensure your AI assets are managed securely.
- **Scalability:** Enhanced compute options such as ZeroGPU enable high-performance model training, deployment, and sharing.
- **Analytics & Control:** Monitor usage with detailed analytics dashboards and centrally manage tokens and billing for your organization.
- **Collaboration:** Private dataset viewers and resource groups make working together across teams efficient and seamless.

Pricing starts at $20 per user per month for teams, with customizable enterprise contracts available.

---

## Company Culture

Hugging Face fosters an inclusive, open, and ethical AI culture. The company thrives on:

- **Open Source Ethos:** Encouraging transparency, sharing, and collaboration as the path to innovation.
- **Community Focus:** Supporting a global network of developers and researchers to drive AI progress collectively.
- **Diversity of Modalities:** Committed to advancing AI across text, vision, audio, and other data types.
- **Empowerment:** Helping users—from beginners to experts—build portfolios, showcase their work, and accelerate their careers.
- **Scientific Excellence:** A talented team pushing the boundaries of machine learning and exploring new frontiers.

---

## Who Are Our Customers?

- **Machine Learning Engineers & Researchers** looking for state-of-the-art models and datasets.
- **Developers & Startups** building AI-powered applications quickly and cost-effectively.
- **Enterprises** seeking secure, scalable AI platforms with industry-compliant features.
- **Educators & Students** learning and experimenting with AI using industry-leading tools.
- **AI Enthusiasts & Innovators** connecting with a vibrant community and sharing solutions worldwide.

---

## Careers at Hugging Face

Hugging Face is actively growing its team of passionate AI professionals. Career opportunities are ideal for innovators who want to:

- Work at the forefront of AI research and deployment.
- Collaborate in an open, inclusive community environment.
- Shape the future of open-source machine learning tools and platforms.
- Drive ethical AI development and positive global impact.

Visit the Hugging Face **[Careers page](https://huggingface.co/careers)** to explore job openings and join the AI revolution.

---

## Join Us

Whether you are a researcher, developer, enterprise leader, or AI enthusiast, Hugging Face invites you to:

- **Explore 2M+ models**, 500K+ datasets, and 1M+ AI applications.
- **Create and share** your models, apps, datasets, and demos.
- **Collaborate** with a thriving community powering the AI future.

**Sign up today at** [huggingface.co](https://huggingface.co) and be part of the AI revolution!

---

**Hugging Face — The AI community building the future.**

In [22]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face: Where AI Meets Hugging (and High-Fives!)

---

## Who Are We?

Welcome to **Hugging Face**, the AI community that's literally *building the future*—one model, dataset, and application at a time. It’s like a massive playground for machine learning nerds, scientists, engineers, and AI enthusiasts to share, discover, experiment, and occasionally high-five through code.

Think of us as the social network for AI... but with fewer cat memes and more breakthrough models. We’re the hub where 2 million+ models (and counting) hang out, datasets abound, and AI apps run wild—all open-source and ready for you to dive in.

---

## What’s in the Hug?

### **Models Galore**
From the latest Qwen multimodal big brain brains to mini models that fit in your pocket, dive into 2 million+ open-source machine learning models. Image? Text? Audio? Video? 3D? You name it, we got it—and folks love to share. Want to turn a photo 360°? We have a model for that. Fancy generating videos from text prompts? Yep, that too.

### **Datasets (Data, Data Everywhere!)**
Half a million+ datasets ready for your next big AI fireworks show. Filter, search, and collaborate on datasets that will power your training runs and push the boundaries of what AI can do.

### **Spaces: Your AI Sandbox**
Create and share AI apps without boilerplate headaches. Want to demo your new text-to-speech wizardry or a nifty image generator? Spaces is your stage. And yes, it’s all running on our slick compute system, ZeroGPU, giving you blazing performance without leaving your chair.

---

## The Culture: Join the AI Revolution (Robes Optional)

At Hugging Face, culture isn’t just a buzzword—it’s our secret sauce. We are:

- **Open & Ethical**: Building AI with transparency and responsibility because Skynet isn’t on our to-do list.
- **Community-Driven**: From beginners making their first model to top-tier AI scientists pushing boundaries—we welcome all.
- **Collaborative & Supportive**: Like a giant AI family—except cooler and with fewer awkward reunions.
- **Fast Movers & Innovators**: We keep it fresh and cutting-edge, with a talented science team exploring AI’s wild frontiers.

If you’re curious, passionate, and a little bit quirky (especially about AI), you’ll feel right at home!

---

## For Companies: Power Your AI with Hugging Face Enterprise

Businesses, listen up: Want to boost your AI game with **enterprise-grade security, dedicated support, and scalable performance**? Get on the Team or Enterprise Hub.

- Single Sign-On (SSO)
- Granular access control & audit logs
- Extra storage & compute on tap (hello, ZeroGPU Quota Boost!)
- Analytics to spy on your own AI usage (no judgment)
- Flex contracts for any size squad

Because AI shouldn’t just be smart—it should be secure and easy to manage too.

---

## Careers: Ready to Hug Your Future?

We’re growing faster than an AI training loop! Join us if you want to:

- Work alongside some of the brainiest AI minds on the planet
- Contribute to open source projects that actually change the world
- Enjoy a culture that celebrates learning, sharing, and lots of virtual hugs (and maybe snacks)
- Help build an AI future that's open and ethical

Check our careers page and come hug the future with us!

---

## Why Hugging Face?

Because AI is better when it’s friendly, open, and collaborative. And also because “hugging” sounds way nicer than “command line” all day.

---

> **Join the  AI revolution — bring your models, your data, your wild ideas — and get ready to transform the future with Hugging Face!**  
>  
> _P.S. Bonus points if you bring your own jokes about robots._

---

### Find us online

- **Explore models & datasets:** [huggingface.co](https://huggingface.co)
- **Join the community:** GitHub | Twitter | Discord | LinkedIn
- **Careers:** [huggingface.co/careers](https://huggingface.co/careers)

---

*Hugging Face: Where the future of machine learning gets a warm, friendly squeeze.* 🤗

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>